In [2]:
import pandas as pd

In [3]:
result_dir = "../results/processedData/normalized_mean_final/"

# produced by dropseq_final_atlas_analysis.R
gene_counts_normalized = "../results/processedData/genes_to_cell_type_table.tsf"
# produced by "uknown"
annotation_file = "../data/hvaepLRv2_kegg_go.tsv"

gene_counts_normalized_table = pd.read_table(gene_counts_normalized)
# If already present:
#annotation_table = pd.read_table("../results/processedData/hvaep_uniprot_kegg_go.tsf")
#annotation_table.columns = ["ID","SP","UniProt","KEGG","KO","GO"]

In [ ]:
gene_counts_normalized_table.head()

In [ ]:
# execute this section if there is still the transcriptome id in use (marked by the "T" in the identifier)
# t_to_g_id = lambda x: "HVAEP1-" + "G"+ x.split(".T")[1].split(".")[0]
# annotation_table.ID = annotation_table.ID.apply(t_to_g_id)
# annotation_table.head()

In [ ]:
# annotation_table.to_csv(result_dir + "/hvaep_uniprot_kegg_go.tsf", sep="\t", index=False)

In [ ]:
annotation_table = pd.read_csv(result_dir+"/hvaep_uniprot_kegg_go.tsf",sep="\t")

In [ ]:
print("Length: {}, Length without duplicates: {}".format(len(annotation_table), len(annotation_table['ID'].unique())))

In [ ]:
print("Length: {}, Length without duplicates: {}".format(len(gene_counts_normalized_table.index), len(gene_counts_normalized_table.index.unique())))

In [ ]:
gene_counts_normalized_table["ID"] = gene_counts_normalized_table.index
normalized_gene_data = gene_counts_normalized_table.merge(annotation_table, on="ID")

# combine both dataframes
merged_df = pd.merge(annotation_table,gene_counts_normalized_table, left_on=["ID"], 
               right_on=["ID"],
               how='outer', indicator=True)
# 
combined_gene_table = pd.concat([merged_df.query('_merge == "right_only"'),normalized_gene_data])
combined_gene_table = combined_gene_table.sort_values(by='ID')

cols = list(gene_counts_normalized_table.columns)
cols.remove("ID")
cols.append("UniProt")
#Cluster ID	Gene ID|Swissprot Annotationb
with open(result_dir + "hvaep_cell_type_to_gene_cluster_table.tsf", 'w') as output_file:
    output_file.write("Cluster ID\tGene ID|Swissprot Annotation\n")
    for identifier in combined_gene_table.ID:
        temp_df = combined_gene_table[combined_gene_table.ID == identifier][cols]
        tdf = temp_df.drop("UniProt", axis=1)
        mean_for_outsort = tdf.T.mean()
        
        for col in temp_df:
            if col != "UniProt":
                # apply filter to discbard low read counts ? 
                if temp_df[col].values[0] > mean_for_outsort.values[0]:
                #if temp_df[col].values[0] > 1:
                    if type(temp_df["UniProt"].values[0]) == str:
                        output_file.write(col + "\t" + identifier + "|"+ temp_df["UniProt"].values[0] + "\n")
                    else:
                        output_file.write(col+ "\t" + identifier + "\n") 


In [ ]:
gene_to_celltype_table = pd.read_table(result_dir + "hvaep_cell_type_to_gene_cluster_table.tsf")
gene_to_celltype_table.head()

In [ ]:
gene_to_celltype_table = pd.read_table(result_dir + "hvaep_cell_type_to_gene_cluster_table.tsf")
print("[+] Length cell type to gene cluster table after filtering for mean values: {}".format(len(gene_to_celltype_table["Gene ID|Swissprot Annotation"].drop_duplicates())))

In [ ]:
gene_counts_normalized_table = pd.read_table(gene_counts_normalized)

gene_counts_normalized_table["ID"] = gene_counts_normalized_table.index
normalized_gene_data = gene_counts_normalized_table.merge(annotation_table, on="ID")

# combine both dataframes
merged_df = pd.merge(annotation_table,gene_counts_normalized_table, left_on=["ID"], 
               right_on=["ID"],
               how='outer', indicator=True)
# 
combined_gene_table = pd.concat([merged_df.query('_merge == "right_only"'),normalized_gene_data])
combined_gene_table = combined_gene_table.sort_values(by='ID')

parsed_ids = []
cols = list(gene_counts_normalized_table.columns)
cols.remove("ID")
cols.append("UniProt")
cols.append("GO")
#Cluster ID	Gene ID|Swissprot Annotation
with open(result_dir + "hvaep_cell_type_to_gene_cluster_table_just_gos.tsf", 'w') as output_file:
    with open(result_dir + "hvaep_uniprot_go_cleaned_table.tsf",'w') as outfile:
        outfile.write("ID\tUniProt\tGO\n")
        output_file.write("Cluster ID\tGene ID|Swissprot Annotation\n")
        for identifier in combined_gene_table.ID:
            temp_df = combined_gene_table[combined_gene_table.ID == identifier][cols]
            tdf = temp_df.drop("UniProt", axis=1)
            tdf = tdf.drop("GO", axis=1)
            mean_for_outsort = tdf.T.mean()
            
            for col in temp_df:
                if col != "UniProt" and col != "GO":
                    # apply filter to discard low read counts ? 
                    if temp_df[col].values[0] > mean_for_outsort.values[0]:
                        if type(temp_df["UniProt"].values[0]) == str and type(temp_df['GO'].values[0]) == str:
                            output_file.write(col + "\t" + identifier + "|"+ temp_df["UniProt"].values[0] + "\n")
                            if identifier not in parsed_ids:
                                outfile.write(identifier+"\t"+temp_df.UniProt.values[0]+"\t"+temp_df.GO.values[0]+"\n")
                                parsed_ids.append(identifier)

In [ ]:
cleaned_annotation_file = pd.read_csv(result_dir + "hvaep_uniprot_go_cleaned_table.tsf", sep="\t")
cleaned_cluster_file = pd.read_csv(result_dir + "hvaep_cell_type_to_gene_cluster_table_just_gos.tsf", sep="\t")

In [ ]:
print("Length annotation table: {}".format(len(cleaned_annotation_file)))

In [ ]:
print("Length of output table: {}".format(len(cleaned_cluster_file)))

In [ ]:
gene_counts_normalized_table = pd.read_table(gene_counts_normalized)

gene_counts_normalized_table["ID"] = gene_counts_normalized_table.index
normalized_gene_data = gene_counts_normalized_table.merge(annotation_table, on="ID")

# combine both dataframes
merged_df = pd.merge(annotation_table,gene_counts_normalized_table, left_on=["ID"], 
               right_on=["ID"],
               how='outer', indicator=True)
# 
combined_gene_table = pd.concat([merged_df.query('_merge == "right_only"'),normalized_gene_data])
combined_gene_table = combined_gene_table.sort_values(by='ID')

with open(result_dir + "hvaep_uniprot_go_full_table.tsf",'w') as outfile:
    outfile.write("ID\tUniProt\tGO\n")
    for identifier in combined_gene_table.ID:
        temp_df = combined_gene_table[combined_gene_table.ID == identifier]
        if type(temp_df.UniProt.values[0]) == str and type(temp_df.GO.values[0]) == str:
            #ID 	SP 	UniProt 	KEGG 	KO 	GO
            outfile.write(temp_df.ID.values[0]+"\t"+temp_df.UniProt.values[0]+"\t"+temp_df.GO.values[0]+"\n")
        elif type(temp_df.UniProt.values[0]) != str and type(temp_df.GO.values[0]) == str:
            outfile.write(temp_df.ID.values[0]+"\t"+''+"\t"+temp_df.GO.values[0]+"\n")
        elif type(temp_df.UniProt.values[0]) == str and type(temp_df.GO.values[0]) != str:
            outfile.write(temp_df.ID.values[0]+"\t"+temp_df.UniProt.values[0]+"\t"+''+"\n")
        else:
            outfile.write(temp_df.ID.values[0]+"\t"+''+"\t"+''+"\n")

# Change deseq2 table header

## No need to execute for the current run -> analysis based on genes.results and not on isoforms.results

In [ ]:
#df = pd.read_csv("../data/hydra_all_counts.tsv", sep="\t")
#df.head()
#df.target_id = df.target_id.apply(lambda x: "HVAEP1-" + x.split(".")[1].replace('T','G'))
#df.head()

In [4]:
df = pd.read_csv("../results/deseq2_rsem_final/hydra_all_counts.tsv", sep="\t")
df.gene_id = df.gene_id.apply(lambda x: "HVAEP1-" + x.split(".")[1])
df.to_csv("../results/deseq2_rsem_final/hydra_all_counts_t_to_g.tsv", sep="\t", header=True, index=False)

df = pd.read_csv("../results/deseq2_rsem_final/hydra_all_deseq2_lg2.tsv", sep="\t")
df.ID = df.ID.apply(lambda x: "HVAEP1-" + x.split(".")[1])
df.to_csv("../results/deseq2_rsem_final/hydra_all_deseq2_lg2_t_to_g.tsv", sep="\t", header=True, index=False)

df = pd.read_csv("../results/deseq2_rsem_final/hydra_all_deseq2_pvalues.tsv", sep="\t")
df.ID = df.ID.apply(lambda x: "HVAEP1-" + x.split(".")[1])
df.to_csv("../results/deseq2_rsem_final/hydra_all_deseq2_pvalues_t_to_g.tsv", sep="\t", header=True, index=False)
df.head()

,ID,EcoKD1_Eco1KD_B8_vs_control_B8,HydraAHL_3OC12_vs_control,HydraAHL_3OHC12_vs_control,HydraRecolonization_Conventionalized_vs_GF,HydraRecolonization_Conventionalized_vs_Wild,HydraRecolonization_Cvbct_vs_GF,HydraRecolonization_Cvbct_vs_Wild,HydraRecolonization_GF_vs_Wild,HydraRecolonization_Wild_vs_GF,HydraTemperature_08°C_vs_18°C,HydraTemperature_12°C_vs_18°C,HydraTemperature_22°C_vs_18°C
0,HVAEP1-G011796,0.000000e+00,2.097391e-02,0.005616,6.938323e-12,0.719574,3.495337e-12,0.743589,2.917355e-15,2.917355e-15,7.784104e-127,2.157681e-116,0.000140
1,HVAEP1-G004481,1.570212e-125,9.564648e-01,0.869496,1.403074e-01,0.986393,9.613706e-01,0.192771,1.109966e-01,1.109966e-01,3.601480e-01,8.891964e-01,0.206467
2,HVAEP1-G000426,3.792892e-125,3.433568e-12,0.000072,4.477707e-01,0.807729,1.428917e-01,0.160880,9.583365e-01,9.583365e-01,5.984516e-07,9.837816e-02,0.016600
3,HVAEP1-G004333,2.555443e-121,4.832504e-01,0.908664,4.844330e-08,0.960255,3.571913e-03,0.176727,2.542949e-07,2.542949e-07,2.290666e-04,3.824062e-02,0.540132
4,HVAEP1-G000462,3.205368e-106,1.949807e-01,0.783110,2.663901e-16,0.711927,1.734126e-02,0.000097,8.173378e-13,8.173378e-13,6.576982e-03,4.589599e-03,0.000003
